### Homework 9: Serverless Deep Learning

In [18]:
import onnxruntime as ort
from io import BytesIO
from urllib import request
import numpy as np

from PIL import Image

#### Question 1 — look at the ONNX graph

Load the model with ONNX Runtime and look at `session.get_inputs()` and
`session.get_outputs()`. What is the output node name?

The input node is `input` and has shape `(batch, 3, 200, 200)`. The output is a
single float: the probability of the `straight` class (`straight=1`).

In [19]:

model_name = "hair_classifier_v1.onnx"

session = ort.InferenceSession(model_name, providers=["CPUExecutionProvider"])
input = session.get_inputs()[0]
output = session.get_outputs()[0]
print(f'Input: {input}')
print(f'Output: {output}')
print(f'Input name: {input.name}')
print(f'Output name: {output.name}')


Input: NodeArg(name='input', type='tensor(float)', shape=['s77', 3, 200, 200])
Output: NodeArg(name='output', type='tensor(float)', shape=['s77', 1])
Input name: input
Output name: output


#### Image preprocessing

In [20]:
def download_image(url):
    with request.urlopen(url, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")


def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return np.transpose(array, (2, 0, 1))[None, ...]

#### Question 2 — target size

What target size does `prepare_image` use?

In [21]:
target_size = (200, 200)
print(f'Target size: {target_size}')

Target size: (200, 200)


#### Question 3 — normalize input

Download `sample.jpeg`, run `prepare_image`, and report the first value of the
R channel (`tensor[0, 0, 0, 0]`) rounded to three decimal places. The grader
accepts an absolute error of `0.01`.

In [23]:
image_url = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
image = download_image(image_url)
image_prepared = prepare_image(image)
first_r = round(image_prepared[0, 0, 0, 0], 3)
print(first_r)

-1.056


#### Question 4 — local ONNX inference

Run the session with:

```python
import onnxruntime as ort

session = ort.InferenceSession(
    "hair_classifier_v1.onnx",
    providers=["CPUExecutionProvider"],
)
output = session.run(
    ["output"],
    {"input": prepare_image(download_image(SAMPLE_URL))},
)[0]
```

Report the output probability rounded to three decimal places. The grader
accepts an absolute error of `0.02`. Because the model, image, and preprocessing
are checksummed and CPU inference is used, this is no longer a choice between
platform-dependent options.

In [28]:
output = session.run(
    ["output"],
    {"input": prepare_image(download_image(image_url))},
)[0]

probability = round(float(output.item()), 3)

print(probability)

0.728


#### Lambda handler

In [32]:
! python smoke_test.py

{'straight_probability': 0.727697, 'straight': True}


#### Question 5 — look at the Lambda configuration

Which runtime base image is declared in the Dockerfile?

In [30]:
runtime_base_image = 'public.ecr.aws/lambda/python:3.13'
print(f'Runtime base image: {runtime_base_image}')

Runtime base image: public.ecr.aws/lambda/python:3.13


#### Question 6 — invoke the container

Inspect the JSON string in the Lambda response body and report
`straight_probability` rounded to three decimal places. The grader accepts an
absolute error of `0.02`. It must match Question 4 because both paths use the
same model, image, preprocessing, and CPU execution provider.

In [33]:
straight_probability = round(0.727697, 3)
print(f'Straight probability: {straight_probability}')

Straight probability: 0.728
